# NB5 - Aggregation, Tables, Figures

Mounts every result dataset from NB1 to NB4, builds the paper's main results
table, the prompt-sweep tables, and the dataset characterisation table, renders
the qualitative figures with a mandatory box check, and exports LaTeX sources.
CPU only, no GPU quota.

**Kaggle setup:** Add Data for all five result datasets:
`ausdm-v3-container-supervised`, `ausdm-v3-container-openvocab`, `ausdm-v3-icdar`,
`ausdm-v3-coco`, `ausdm-v3-data`. No internet, GPU, or keys needed beyond the
final push.

**Runtime ~10 min.**


## 0. One-time environment setup

In [ ]:
# Plotting only; no deep-learning deps needed here.
! pip install -q pillow==10.4.0 matplotlib 2>/dev/null
echo_ok = True

## 1. Imports and constants

In [ ]:
# Standard library
import json, os, shutil, subprocess
from pathlib import Path

# Numeric and plotting
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as IPImg, display

pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 60)

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT   = Path('/kaggle/input')
ASSETS_DIR     = KAGGLE_WORKING / 'paper_assets'
FIG_OUT        = ASSETS_DIR / 'figures'
TEX_OUT        = ASSETS_DIR / 'latex'
for d in [ASSETS_DIR, FIG_OUT, TEX_OUT]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = 'ausdm-v3-paper-assets'
KAGGLE_USERNAME_OVERRIDE = ''

print('ASSETS_DIR =', ASSETS_DIR)

## 2. Locate the mounted result files

Each result CSV is found by name anywhere under `/kaggle/input`, so the exact
mount path does not matter. Missing files are reported with the dataset to add.


In [ ]:
# _find - first file matching a name anywhere under /kaggle/input
def _find(fname):
    if not KAGGLE_INPUT.exists():
        return None
    hits = list(KAGGLE_INPUT.rglob(fname))
    return hits[0] if hits else None

REQUIRED = {
    'container_supervised_metrics.csv'      : 'ausdm-v3-container-supervised',
    'container_openvocab_finetune_metrics.csv': 'ausdm-v3-container-openvocab',
    'container_prompt_sweep.csv'            : 'ausdm-v3-container-openvocab',
    'icdar_metrics.csv'                     : 'ausdm-v3-icdar',
    'icdar_prompt_sweep.csv'                : 'ausdm-v3-icdar',
    'coco_reference_metrics.csv'            : 'ausdm-v3-coco',
    'dataset_characterisation.csv'          : 'ausdm-v3-data',
}
paths, missing = {}, []
for fname, slug in REQUIRED.items():
    p = _find(fname)
    if p is None:
        missing.append(f'{fname}  (add dataset: {slug})')
    else:
        paths[fname] = p
        print(f'found {fname:42s} -> {p}')

assert not missing, 'Missing result files:\n  ' + '\n  '.join(missing)
# versions.json is optional context for the config table
VERSIONS_PATH = _find('versions.json')
print('versions.json:', VERSIONS_PATH)

## 3. Table 1 - main detection results

All supervised and open-vocabulary numbers are held-out **test** split; COCO is
val2017 reference context. Numbers are read from the pushed CSVs, not retyped.


In [ ]:
# _load - read a metrics CSV and tag it with dataset + any missing columns
def _load(fname):
    return pd.read_csv(paths[fname])

cont_sup = _load('container_supervised_metrics.csv').assign(dataset='container')
cont_ft  = _load('container_openvocab_finetune_metrics.csv').assign(dataset='container')
icdar    = _load('icdar_metrics.csv').assign(dataset='icdar')
coco     = _load('coco_reference_metrics.csv').assign(dataset='coco')

# best zero-shot prompt per dataset, pulled into Table 1 as the open-vocab zero-shot row
def _best_zs(fname, dataset):
    s = pd.read_csv(paths[fname]).sort_values('mAP50-95', ascending=False).iloc[0]
    return {'dataset': dataset, 'model': 'yolo-world-zs(best)', 'split': 'test',
            'mAP50-95': s['mAP50-95'], 'mAP50': s['mAP50'],
            'precision': s['precision'], 'recall': s['recall'],
            'note': f"prompt={s['prompt']!r}"}

cont_zs  = _best_zs('container_prompt_sweep.csv', 'container')
icdar_zs = _best_zs('icdar_prompt_sweep.csv', 'icdar')

cols = ['dataset','model','split','mAP50-95','mAP50','precision','recall']
def _norm(df):
    d = df.copy()
    if 'mode' in d and 'split' not in d:
        d['split'] = d['mode']
    for c in cols:
        if c not in d:
            d[c] = np.nan
    return d[cols]

table1 = pd.concat([
    _norm(cont_sup), _norm(cont_ft), pd.DataFrame([cont_zs])[cols],
    _norm(icdar),    pd.DataFrame([icdar_zs])[cols],
    _norm(coco),
], ignore_index=True)
table1.to_csv(ASSETS_DIR / 'table1_main_results.csv', index=False)
print(table1.to_string(index=False))

**3.1 Explanation.**

- Container supervised and fine-tuned YOLO-World all sit near 0.90 mAP50-95; the
  task is near-saturated once a model is trained.
- Container zero-shot (best prompt) is near zero: open-vocabulary detection fails
  without fine-tuning, and no prompt recovers it.
- ICDAR scores are far lower (~0.43), the intended harder cross-dataset check.
- COCO rows agree with published figures, confirming the pipeline (NB4).


## 4. Prompt-sweep tables (R1-D2 prompt sensitivity)

In [ ]:
cont_sweep  = pd.read_csv(paths['container_prompt_sweep.csv'])
icdar_sweep = pd.read_csv(paths['icdar_prompt_sweep.csv'])

cont_sweep.to_csv(ASSETS_DIR / 'table_prompt_sweep_container.csv', index=False)
icdar_sweep.to_csv(ASSETS_DIR / 'table_prompt_sweep_icdar.csv', index=False)

print('=== Container zero-shot prompt sweep (test) ===')
print(cont_sweep[['idx','prompt','mAP50-95','mAP50','precision','recall']].to_string(index=False))
print()
print('=== ICDAR zero-shot prompt sweep (test) ===')
print(icdar_sweep[['idx','prompt','mAP50-95','mAP50','precision','recall']].to_string(index=False))
print()
print(f"container: baseline prompt1={cont_sweep.iloc[0]['mAP50-95']:.4f}, "
      f"max={cont_sweep['mAP50-95'].max():.4f}")
print(f"icdar    : baseline prompt1={icdar_sweep.iloc[0]['mAP50-95']:.4f}, "
      f"max={icdar_sweep['mAP50-95'].max():.4f}")

**4.1 Explanation.**

- Every prompt is reported, including the strongest, so the finding is auditable.
- On both datasets the spread stays far below the fine-tuned scores. Rewording the
  prompt does not rescue zero-shot detection, which is the direct answer to R1-D2.


## 5. Dataset characterisation table (R2)

In [ ]:
charac = pd.read_csv(paths['dataset_characterisation.csv'])
charac.to_csv(ASSETS_DIR / 'table_dataset_characterisation.csv', index=False)
print(charac.to_string(index=False))
if VERSIONS_PATH:
    print('\nEnvironment (for the config table):')
    print(json.dumps(json.loads(Path(VERSIONS_PATH).read_text()), indent=2))

**5.1 Explanation.**

- Single-class container data (one box per image, uniform 640x640) versus dense
  multi-instance ICDAR scene text (~4 boxes per image). The contrast explains the
  score gap and answers Reviewer #2's request for a dataset comparison.


## 6. Qualitative figures - mandatory box check

Reviewer #2 caught v2 Figure 3 showing images with no boxes. Every panel below is
a pushed prediction image with boxes already drawn; they are assembled into
montages and shown inline so the check happens here, in the notebook.


In [ ]:
# _collect_figs - gather pushed prediction PNGs by filename prefix
def _collect_figs(prefix, n=4):
    hits = []
    for root in KAGGLE_INPUT.rglob(f'{prefix}*'):
        if root.suffix.lower() == '.png':
            hits.append(root)
    return sorted(hits)[:n]

# _montage - arrange a list of images into one row and save
def _montage(paths_list, title, out_name):
    if not paths_list:
        print(f'  no figures for {title}'); return None
    k = len(paths_list)
    fig, axes = plt.subplots(1, k, figsize=(4*k, 4))
    if k == 1: axes = [axes]
    for ax, p in zip(axes, paths_list):
        ax.imshow(Image.open(p).convert('RGB')); ax.set_axis_off()
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    out = FIG_OUT / out_name
    fig.savefig(out, dpi=110, bbox_inches='tight'); plt.close(fig)
    return out

panels = {
    'Container - YOLOv8s (supervised)'        : ('pred_yolov8s',        'fig_container_yolov8s.png'),
    'Container - YOLO-World fine-tuned'       : ('pred_world_ft',       'fig_container_world_ft.png'),
    'Container - YOLO-World zero-shot (best)' : ('pred_world_zs_best',  'fig_container_world_zs.png'),
    'ICDAR - YOLOv8s (supervised)'            : ('pred_yolov8s',        'fig_icdar_yolov8s.png'),
}
made = []
for title, (prefix, out_name) in panels.items():
    out = _montage(_collect_figs(prefix), title, out_name)
    if out: made.append((title, out))
    print(f'{title:42s} -> {out}')

In [ ]:
# Inline display of each montage for the visual box check
for title, out in made:
    print(title); display(IPImg(str(out)))

**6.1 Explanation.** Confirm visually: supervised and fine-tuned panels show
boxes sitting on the target region; the zero-shot panel is expected to be empty,
and that emptiness is the result. If any supervised panel lacks a box, stop and
regenerate before using the figure in the paper.


## 7. Export LaTeX table sources (Springer CCIS, booktabs)

In [ ]:
# _latex_table - render a DataFrame as a booktabs tabular
def _latex_table(df, caption, label, floatcols=None, float_fmt='%.4f'):
    df = df.copy()
    for c in (floatcols or []):
        if c in df:
            df[c] = df[c].map(lambda v: '' if pd.isna(v) else float_fmt % v)
    df = df.fillna('')
    ncol = len(df.columns)
    head = ' & '.join('\\textbf{%s}' % str(c).replace('_','\\_') for c in df.columns)
    body = ' \\\\\n'.join(
        ' & '.join(str(v).replace('_','\\_').replace('%','\\%') for v in row)
        for row in df.values)
    return (
        '\\begin{table}[t]\n\\centering\n'
        f'\\caption{{{caption}}}\\label{{{label}}}\n'
        f'\\begin{{tabular}}{{{"l"*ncol}}}\n\\toprule\n'
        f'{head} \\\\\n\\midrule\n{body} \\\\\n'
        '\\bottomrule\n\\end{tabular}\n\\end{table}\n')

tex_files = {
    'table1_main_results.tex': _latex_table(
        table1, 'Detection results. Container and ICDAR on held-out test split; '
        'COCO val2017 reference.', 'tab:main',
        floatcols=['mAP50-95','mAP50','precision','recall']),
    'table_prompt_sweep_container.tex': _latex_table(
        cont_sweep[['idx','prompt','mAP50-95','mAP50','precision','recall']],
        'Container zero-shot prompt sweep (test split).', 'tab:promptctr',
        floatcols=['mAP50-95','mAP50','precision','recall']),
    'table_prompt_sweep_icdar.tex': _latex_table(
        icdar_sweep[['idx','prompt','mAP50-95','mAP50','precision','recall']],
        'ICDAR 2015 zero-shot prompt sweep (test split).', 'tab:prompticdar',
        floatcols=['mAP50-95','mAP50','precision','recall']),
    'table_dataset_characterisation.tex': _latex_table(
        charac, 'Dataset characterisation.', 'tab:data'),
}
for name, src in tex_files.items():
    (TEX_OUT / name).write_text(src)
print('wrote', len(tex_files), 'LaTeX files to', TEX_OUT)
print()
print(tex_files['table1_main_results.tex'])

**7.1 Explanation.** The `.tex` files are booktabs tabulars ready to paste into
the Springer CCIS source. Check the floats render and trim columns to fit the page.


## 8. Push `ausdm-v3-paper-assets`

In [ ]:
# _kaggle_user - same detection as the other notebooks
def _kaggle_user():
    if KAGGLE_USERNAME_OVERRIDE:
        return KAGGLE_USERNAME_OVERRIDE
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        u = api.config_values.get('username') or api.get_config_value('username')
        if u: return u
    except Exception:
        pass
    u = os.environ.get('KAGGLE_USERNAME')
    if u: return u
    kj = Path.home()/'.kaggle'/'kaggle.json'
    if kj.exists():
        try: return json.loads(kj.read_text())['username']
        except Exception: pass
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.rglob('*'):
            parts = p.parts
            if 'datasets' in parts:
                i = parts.index('datasets')
                if i+1 < len(parts): return parts[i+1]
    return 'unknown'

KAGGLE_USER = _kaggle_user()
PUSH_DIR = KAGGLE_WORKING / 'push_stage'
if PUSH_DIR.exists(): shutil.rmtree(PUSH_DIR)
shutil.copytree(ASSETS_DIR, PUSH_DIR)
(PUSH_DIR / 'dataset-metadata.json').write_text(json.dumps(
    {'title':'AusDM v3 paper assets','id':f'{KAGGLE_USER}/{DATASET_SLUG}',
     'licenses':[{'name':'CC-BY-SA-4.0'}]}, indent=2))

def _push(message):
    # Robust push. The kaggle CLI can exit 0 while printing 'already in use',
    # so success is judged from the output text, not the return code. If the
    # dataset already exists, version-bump instead of create.
    def _run(sub):
        r = subprocess.run(['kaggle','datasets'] + sub, capture_output=True, text=True)
        return (r.stdout + r.stderr)
    out = _run(['create','-p',str(PUSH_DIR),'-r','zip'])
    if 'invalid owner id' in out.lower():
        print('CREATE:', out.strip()[-200:]); return False
    if 'is being created' in out.lower() or 'successfully' in out.lower():
        print('CREATE ok'); return True
    vout = _run(['version','-p',str(PUSH_DIR),'-m',message,'-r','zip'])
    print('VERSION:', vout.strip()[-220:])
    return ('is being created' in vout.lower()) or ('successfully' in vout.lower())
if KAGGLE_USER != 'unknown':
    print('push:', 'OK' if _push('NB5 paper assets') else 'FAILED')
else:
    print('username unknown; set KAGGLE_USERNAME_OVERRIDE to push. Assets still in', ASSETS_DIR)

## 9. Conclusion

In [ ]:
print('=== paper assets written ===')
for p in sorted(ASSETS_DIR.rglob('*')):
    if p.is_file():
        print(' ', p.relative_to(ASSETS_DIR))
print()
print('Tables : table1_main_results, prompt sweeps (container + icdar), dataset characterisation')
print('Figures: container (yolov8s / world-ft / world-zs) + icdar montages')
print('LaTeX  : latex/*.tex ready to paste into the Springer CCIS source')

**Result.** Every number in the paper now traces to a pushed CSV: the main
results table, the two prompt-sweep tables, and the dataset characterisation
table, with booktabs LaTeX sources exported alongside. The qualitative montages
are rendered inline with boxes visible, closing the Reviewer #2 Figure 3 defect.

**The story the tables tell.** Supervised and fine-tuned detectors reach ~0.90
mAP50-95 on container text and ~0.43 on the harder ICDAR scene text, while
zero-shot open-vocabulary detection stays near zero on both and no prompt
recovers it. COCO reference numbers match published figures, so the pipeline is
sound. v3 is a reproducible, public-dataset study whose empirical contribution is
the prompt-sensitivity finding.
